In [1]:
import os
os.chdir('..')
from gpu_management import set_gpus

os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'
# os.environ['CUDA_VISIBLE_DEVICES'] = '4'
set_gpus(1, forcing=True)

from data import DL17Lands
from card_utils import get_card_image

dataloader = DL17Lands('FDN')

Loading FDN
-----------
Loading card data: 0.195638s
Making extension dataframe: 1.539742s
##################################################


In [2]:
dl_eoe = DL17Lands('EOE')

Loading EOE
-----------
Loading card data: 0.000053s
Making extension dataframe: 0.286043s
##################################################


In [3]:
import polars as pl

dl_eoe.cards.filter(pl.col('name') == 'Quantum Riddler')['oracle'][0]

'[NAME] Quantum Riddler | [MC] {3}{U}{U} | [CMC] 5 | [COLOR] U | [RARITY] mythic | [TYPE] Creature | [SUBTYPE] Sphinx | [P/T] 4/6 | [KEYWORDS] Flying, Warp | [EFFECTS] Flying <NL> When this creature enters, draw a card. <NL> As long as you have one or fewer cards in hand, if you would draw one or more cards, you draw that many cards plus one instead. <NL> Warp {1}{U}'

In [4]:
packs = dataloader.drafts.filter((pl.col('pick_number') == 0) & (pl.col('pack_number') == 0)).collect()['pack']

In [5]:
from IPython.display import display, HTML
import numpy as np

GREEN = "#7FFF55"
RED = "#FF5555"
BLUE = "#557FFF"
def solid_border(color, width=2):
    return f"border: {width}cqw solid {color};" 
    
def striped_border(colors, width=2):
    step = 100 / len(colors)
    style = f"""
        border: {width}cqw solid transparent;
        border-image: linear-gradient(
            to bottom right,
            {colors[0]} {step}%,
    """
    for i, color in enumerate(colors[1:-1]):
        style += f"""
            {color} {step*(i+1)}%,
            {color} {step*(i+2)}%,
        """
    style += f"""
            {colors[-1]} {100-step}%
        );
        border-image-slice: 1;
    """
    return style

def show_pack(pack, pick=-1, pack_size=15, stats=None, border=3):
    width = 99 / pack_size
    height = width * 88 / 63
    
    images = []
    min_pick, max_pick = -1, -1
    if stats is not None:
        valid = ~np.isnan(stats)
        min_pick = np.argmin(stats[valid])
        max_pick = np.argmax(stats[valid])
    def render_card(i, card_id):
        result = dataloader.all_cards.filter(pl.col('id') == card)
        assert len(result) == 1
        card_url = get_card_image(result['name'][0])
        colors = []
        if i == pick:
            colors.append(GREEN)
        if i == min_pick:
            colors.append(RED)
        if i == max_pick:
            colors.append(BLUE)
            
        if len(colors) == 1:
            style = solid_border(colors[0], border)
        elif len(colors) > 1:
            style = striped_border(colors, border)
        else:
            style = solid_border("transparent", border)
        return f"""<div style="container-type: size;
            display: inline-block;
            background-image: url('{card_url}');
            background-size: cover;
            width: {width}cqw;
            height: {height}cqw;">
<div style="display: inline-block;
            width: {100 - 2*border}cqw;
            height: calc(100cqh - {2*border}cqw);
            {style}"></div></div>"""
    if stats is not None:
        order = np.argsort(-stats)
    else:
        order = np.arange(pack_size-1)
        order = np.concatenate([
            [pick],
            np.where(order < pick, order, order+1)
        ])
    for i in order:
        card = pack[i.item()]
        if card == 0: continue
        images.append(render_card(i, card))
            
    return f"""<div style="container-type: inline-size; margin: 0; padding: 0; height: {height}cqw;">{"".join(images)}</div>"""
display(HTML(show_pack(packs[1], 12)))

In [6]:
drafts = dataloader.drafts.group_by('draft_id').agg(pl.col('pack'), pl.col('pick_id'), pl.col('user_game_win_rate_bucket'), pl.col('event_match_wins'), pl.col('event_match_losses')).collect()
draft = drafts.sample(1)[0]
draft_picks = list(zip(draft['pack'].to_list()[0], draft['pick_id'].to_list()[0]))

In [7]:
draft

draft_id,pack,pick_id,user_game_win_rate_bucket,event_match_wins,event_match_losses
str,"list[array[i32, 15]]",list[i64],list[f32],list[i8],list[i8]
"""4d9061bae81f4ab88193f173a19e89…","[[93863, 93882, … 0], [93865, 93928, … 0], … [93863, 0, … 0]]","[93839, 93888, … 93863]","[0.52, 0.52, … 0.52]","[3, 3, … 3]","[3, 3, … 3]"


In [8]:
pack_size = len(draft_picks) // 3
res = ""
for i_pack in range(3):
    block = ""
    for pack, pick in draft_picks[i_pack * pack_size:(i_pack+1) * pack_size]:
        block += show_pack(pack, pack.index(pick), pack_size)
    res += f"""<div style="flex: 1; margin: 0px; container-type: inline-size;">{block}</div>"""
display(HTML(f"""<div style="display: flex;">{res}</div>"""))

In [9]:
from nlp import Gemma
from data import JaxDraftDataset

data = JaxDraftDataset(
    [dataloader],
    time_window=14,
    seed=42,
    verbose=1
)

Collating FDN took 1.672724s
Cards use:    8.04 KB
Sets use:     1.15 KB
Drafts use:  65.26 MB


In [10]:
def show_pack_jax(pack, pick, pack_size=15, stats=None):
    return show_pack(pack, np.argmax(pack == pick), pack_size=pack_size, stats=stats)

In [11]:
import numpy as np
draft_id = 0
display(HTML(show_pack_jax(data.cards.card_id[data.drafts.packs[draft_id]][0], data.cards.card_id[data.drafts.picks[draft_id]][0])))

In [12]:
display(HTML(show_pack_jax(data.cards.card_id[data.drafts.packs[draft_id]][1], data.cards.card_id[data.drafts.picks[draft_id]][1])))

In [13]:
display(HTML(show_pack_jax(data.cards.card_id[data.drafts.packs[draft_id]][2], data.cards.card_id[data.drafts.picks[draft_id]][2])))

In [14]:
import jax

def show_draft_jax(draft, cards, vert=False):
    pack_size = (draft.packs[0] > 0).sum()
    res = ""
    for i_pack in [2]: #range(3):
        block = ""
        for i_pick in range(i_pack * pack_size, (i_pack+1) * pack_size):
            block += show_pack_jax(cards.card_id[draft.packs[i_pick]], cards.card_id[draft.picks[i_pick]], pack_size=pack_size)
        res += f"""<div style="flex: 1; margin: 0px; container-type: inline-size;">{block}</div>"""
    return f"""<div style="display: {'block' if vert else 'flex'};">{res}</div>"""
display(HTML(show_draft_jax(jax.tree.map(lambda x: x[draft_id], data.drafts), data.cards, True)))

In [15]:
drafts_head = data.drafts[10:20]
drafts_head.player_wr, drafts_head.game_outcome, drafts_head.rank

(Array([0.52, 0.58, 0.58, 0.48, 0.5 , 0.38, 0.58, 0.4 , 0.4 , 0.52],      dtype=float32),
 Array([[3, 3],
        [3, 3],
        [6, 3],
        [1, 3],
        [1, 3],
        [3, 3],
        [7, 2],
        [1, 3],
        [0, 3],
        [5, 3]], dtype=int8),
 Array([4, 3, 2, 3, 0, 4, 4, 2, 3, 4], dtype=int16))

In [16]:
%%time

import pickle
import equinox as eqx

from nlp import get_encoder
from models import get_model
from data import make_dataset_from_args
from experiment import prepare_trainer
from eval import prepare_rankings, card_rankings

path = "experiments/results/SameSet2"
#path = "experiments/results/NewSet"
i_model = 12
with open(f'{path}/results.pkl', 'rb') as f:
    exp_args, results = pickle.load(f)
    results = results[i_model]
    args = results['config']
    print(args)
data_train, data_val, data_test = make_dataset_from_args(args.dataset)

encoder = get_encoder(args.encoder)

data_train.process_data(encoder, args.graph_density, args.graph_type)
data_val.process_data(encoder, args.graph_density, args.graph_type)
data_test.process_data(encoder, args.graph_density, args.graph_type)
data.process_data(encoder, args.graph_density, args.graph_type)

model_params = args.model_params
key, subkey = jax.random.split(jax.random.PRNGKey(args.seed))
model, state = eqx.nn.make_with_state(get_model(args.model))(
    key=subkey, cards=data_train.cards, **model_params
)
params, static = eqx.partition(model, eqx.is_array)
trainer, opt_state, lr_transform, lr_transform_state = prepare_trainer(args, model, data_train.n_steps(args.batch_size))
params, state, opt_state = eqx.tree_deserialise_leaves(
    #path+'/model-02-DraftGraph.eqx',
    path+f'/model-{i_model+1:02}-{args.model}.eqx',
    (params, state, opt_state)
)
key, subkey = jax.random.split(key)
draft_backup = data.drafts
data.drafts = data.drafts[:64]
prepare_rankings(params, static, state, args.batch_size // 32, data, trainer, key)
rankings = card_rankings(params, state, args.batch_size // 32, data, key)
data.drafts = draft_backup
draft_id = 11
# rankings = rank_draft(params, static, state, data, draft_id, key)
rankings.shape, rankings[draft_id,:3,:3]

Namespace(model='DraftTransformer2', model_params={'d_model': 64, 'dropout_p': 0.3, 'n_set_layers': 10, 'n_layers': 5, 'num_heads': 2, 'pred_n_layers': 2}, encoder='gemma', tokenizer='bert', use_meta=False, graph_density=0.01, graph_type='knn', epochs=100, batch_size=256, optimizer='adamw', lr=0.0001, scheduler='warmup_cosine', scheduler_params={'init_mult': '1e-1', 'warmup_steps': 1, 'end_mult': '1e-1', 'n_restarts': 0, 'restart_factor': 1}, loss=['NLL', 'KL'], fine_tune=None, fine_tune_epochs=None, test_frequency=1, seed=42, dataset={'train_set': ['BLB', 'FDN', 'TDM'], 'val_set': ['BLB', 'FDN', 'TDM'], 'test_set': ['BLB', 'FDN', 'TDM'], 'temporal_split': True, 'time_window': 14, 'data_head': -1}, param_count=599745)
Precompiled ranking in 26.476904s
Ranked all drafts in 5.963432s
CPU times: user 3min 2s, sys: 34.4 s, total: 3min 36s
Wall time: 1min 35s


((64, 45, 15),
 Array([[0.5412163 , 0.40273196, 0.39081144],
        [0.5511346 , 0.4991671 , 0.5517266 ],
        [0.5567832 , 0.55844873, 0.51888865]], dtype=float32))

In [17]:
display(HTML(show_pack_jax(data.cards.card_id[data.drafts.packs[draft_id]][14], data.cards.card_id[data.drafts.picks[draft_id]][14], stats=rankings[draft_id,14])))

In [27]:
np.set_printoptions(linewidth=100)
def show_full_draft_jax(draft, cards, outputs, vert=False, first_n=None):
    pack_size = (draft.packs[0] > 0).sum()
    if first_n is None:
        first_n = pack_size
    res = ""
    for i_pack in [2]:#range(3):
        block = ""
        for i_pick in range(i_pack * pack_size, (i_pack+1) * pack_size)[:first_n]:
            block += show_pack_jax(cards.card_id[draft.packs[i_pick]], cards.card_id[draft.picks[i_pick]], 14, stats=outputs[i_pick])
        res += f"""<div style="flex: 1; margin: 0px; container-type: inline-size;">{block}</div>"""
    return f"""<div style="display: {'block' if vert else 'flex'};">{res}</div>"""
draft_id = 8
print(rankings[draft_id,0])
display(HTML(show_full_draft_jax(jax.tree.map(lambda x: x[draft_id], data.drafts), data.cards, rankings[draft_id], vert=True)))

[0.42765218 0.40517285 0.45154664 0.51419604 0.520795   0.47541526 0.45512128 0.40138298 0.52049845
 0.5495894  0.504608   0.5556935  0.518065   0.4720432         nan]


In [28]:
with open("figures/draft.html", "w") as f:
    f.write(show_full_draft_jax(jax.tree.map(lambda x: x[draft_id], data.drafts), data.cards, rankings[draft_id], vert=True))